# TVAC Ethernet Connection and Remote-Control Commissioning

Last updated: 2026-09-17

This notebook drives the `hvc3500` package in this repository through the
commissioning sequence from `TVAC_ETHERNET_PROGRESS.md`: connect, prove the
read-only path, learn the session behaviour, take a snapshot, and only then
make one supervised write.

Run it from the repository root with a Python 3.11+ kernel (the
`formsLabCLI` venv works). Every transaction is also written to `logs/`.

## 0. What the manual establishes

- The controller enclosure RJ45 labelled **Ethernet** reaches two addresses:
  the **Panel IP** (VNC, FTP) and the **CPU IP** (ASCII over TCP). Use the CPU IP.
- Both addresses and the **CPU TCP** port are shown on the HMI under
  *UniApps -> Network -> Ethernet*. The manual's screenshot shows
  `172.16.21.74` (panel) / `172.16.21.75` (CPU) as an **example only**.
- Reads start with `?`, writes with `!`, lines end with CR. Unknown -> `ER`.
- Temperatures are tenths without a decimal point (`855` = 85.5).
- `?ES` returns `ES: N|W|F: <UINT32 mask>`; bit *n* = fault/warning *n*.

## Live results (2026-09-17)

Run against the installed chamber from this PC (10.1.2.200/24 on Ethernet 3):

| step | result |
| --- | --- |
| TCP connect 10.1.2.121:1 | connected |
| `?MC<CR>` | `Manual<CR>` (installed software uses mixed case) |
| `?MC<LF>` | `ER` - CR is the terminator; the manual's `0x13` is a typo |
| `?QQ<CR>` | `ER` |
| lifecycle | persistent and one-shot sessions both OK, ~105 ms/reply |
| all 49 documented reads | answered; raw frames in `logs/20260917_073440_live_readsweep.jsonl` |
| snapshot | Manual / Recovery-Ready / warning bit 16 "Control air low" / all valves+pumps closed / pressure 35.5 (unit TBC) / T0-T15 19-24 |

VNC to the Panel IP works for viewing (`vncviewer64.exe 10.1.2.120:5900`) but
the session is view-only; that is a touchscreen setting, and irrelevant to
automation, which uses this ASCII link.

In [ ]:
import json, os, sys, time
REPO = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'Notebook' else os.getcwd()
sys.path.insert(0, REPO)
from hvc3500 import HVC3500Client, ProtocolError, WriteRefused, protocol as P
from hvc3500.cli import _print_tx
print('hvc3500 package loaded from', REPO)

## 1. Bench configuration

Fill these in from the HMI. Nothing below is guessed. If you do not yet know
the address, run `tools\find_hvc.ps1` in an elevated PowerShell with the cable
plugged into the adapter that links up (this PC: `Ethernet 3`), or use
`python -m hvc3500 discover --subnet 172.16.21 --port <port>` once the PC has
an address on the controller's subnet.

In [ ]:
CPU_IP   = os.environ.get('HVC_HOST', '10.1.2.121')     # CPU IP, confirmed live 2026-09-17 (Panel IP is 10.1.2.120)
CPU_PORT = int(os.environ.get('HVC_PORT', '1'))         # ASCII socket found by port scan 2026-09-17
TIMEOUT  = 2.0
UNITS    = {'pressure': 'unknown', 'temperature': 'unknown'}  # still to read from HMI Adv Settings
ZONES    = {'zone1': 1, 'zone2': 2}                     # shroud/platen mapping still to confirm on HMI
SENSORS  = {f'T{i}': i for i in range(16)}              # T0-T15 answer live; T16-T20 unused

LOG = os.path.join(REPO, 'logs', time.strftime('%Y%m%d_%H%M%S') + '_notebook.jsonl')
os.makedirs(os.path.dirname(LOG), exist_ok=True)
_logf = open(LOG, 'a')
def on_tx(tx):
    _logf.write(tx.as_json() + '\n'); _logf.flush(); _print_tx(tx)
print('endpoint', CPU_IP, CPU_PORT, '-> log', LOG)

## 2. First discriminating test

TCP connect, then `?MC<CR>`. Outcomes:

| result | meaning |
| --- | --- |
| connect refused / timeout | wrong IP/port, path, firewall, or the CPU TCP service is off |
| connected, no reply | wrong terminator or the controller wants one command per connection (re-run with `persistent=False`) |
| `ER` | service is up; installed software differs from the manual |
| `AUTO` / `MANUAL` | proceed |

In [ ]:
c = HVC3500Client(CPU_IP, CPU_PORT, timeout=TIMEOUT, persistent=True, on_transaction=on_tx)
try:
    c.connect(); print('connected')
    print('mode =', c.mode())
except OSError as e:
    print('FAILED:', e)

## 3. Read-only status queries (record the raw frames)

In [ ]:
for name, fn in [('test_status', c.test_status), ('pressure', c.pressure),
                 ('pressure_rate', c.pressure_rate), ('vacuum_setpoint', c.vacuum_setpoint),
                 ('recipe', c.recipe), ('recipe_step', c.recipe_step),
                 ('error_status', c.error_status)]:
    try:
        print(f'{name:>16} = {fn()}')
    except (OSError, ProtocolError) as e:
        print(f'{name:>16} FAILED: {e}')
try:
    c.transact('?QQ')
except ProtocolError as e:
    print('unknown command ->', e)

## 4. Session behaviour and timing

Does the controller hold a TCP session across commands, or expect one
command per connection? What is the reply latency? This decides
`persistent=` and the safe polling interval for `watch`.

In [ ]:
for persistent in (True, False):
    t = HVC3500Client(CPU_IP, CPU_PORT, timeout=TIMEOUT, persistent=persistent)
    ok, err = True, None
    try:
        for _ in range(5):
            t.mode()
    except (OSError, ProtocolError) as e:
        ok, err = False, e
    t.close()
    times = [x.elapsed_ms for x in t.transactions if x.ok]
    print(f'persistent={persistent}: {"OK" if ok else "FAILED " + str(err)}',
          f'({min(times):.0f}-{max(times):.0f} ms)' if times else '')

## 5. Read-only snapshot, compared against the HMI

In [ ]:
snap = c.snapshot(temperatures=list(SENSORS.values()), zones=list(ZONES.values()))
for name, n in SENSORS.items():
    snap[f'temp.{name}'] = snap.get(f'T{n}')
for name, z in ZONES.items():
    snap[f'zone.{name}.setpoint'] = snap.get(f'Z{z}_setpoint')
print(json.dumps(snap, indent=2))
print('\nCompare pressure, temperatures and setpoints with the HMI now. Units assumed:', UNITS)

## 6. One supervised, idempotent write (only after section 5 matches the HMI)

Writes a zone setpoint equal to its current value first (a no-op that proves
the write path), then optionally a small change. Requires an operator at the
chamber. Set `DO_WRITE = True` deliberately.

In [ ]:
DO_WRITE = False
ZONE = ZONES['shroud']
if DO_WRITE:
    current = c.zone_setpoint(ZONE)
    print('no-op write:', c.set_zone_setpoint(ZONE, current))
    # print('small change:', c.set_zone_setpoint(ZONE, current + 0.5))
else:
    print('DO_WRITE is False; skipping. Read-only so far.')

## 7. Non-idempotent commands

`!CS` (start/continue), `!CA`, `!CR`, vent/fill and the valve/pump toggles are
deliberately behind `confirm=True` in the client and are **not** part of
connectivity testing. Use them only inside a supervised, documented test with
the ASCII recipe configured (Advanced Settings -> *Enable ASCII in Recipe*;
recipe vacuum operation = ASCII; trigger = hold).

```python
# c.action('CS', confirm=True)                 # start / continue held recipe step
# c.set_device('OR', True, confirm=True)       # read-first, one toggle, wait, verify
```

In [ ]:
c.close(); _logf.close()
print('closed; transactions logged:', len(c.transactions), '->', LOG)